# Notebook 24 — Parameter-attribution residual audit (reviewer comment A13)

**Author:** Sunita Sahu · **Revision item:** A13 (parameter attribution residual check)

## Reviewer comment

> IDEFICS-9B total measured as 7,495.2 M, but the released model is ~8.9–9B. Perceiver Resampler and gated cross-attention dense layers appear nowhere in Table 2. Re-run the parameter traversal for IDEFICS-9B and Llama-3.2-Vision with an explicit residual check (attributed-component sum vs. checkpoint total).

## What this notebook does

1. **Structure audit (16 transformers-native models).** Each model is instantiated on the `meta` device from its released `config.json` — exact tensor shapes, no weights, seconds per model on a CPU. The revised `vlm_audit` extractor then assigns **every parameter to exactly one component** (`metrics.partition_component_params`) and reports the **residual** = whole-model total − attributed sum.
2. **Checkpoint audit (4 remote-code models).** InternVL3, Molmo, DeepSeek-VL2 and Janus-Pro need custom modelling code; their tensors are attributed directly from the released checkpoint's tensor names (longest-prefix rule, the checkpoint-side equivalent of the partition).
3. **Independent cross-check (all 20).** Totals are read from the released checkpoint shard headers (safetensors JSON headers, or the `data.pkl` index of `.bin` archives via HTTP range requests) and reconciled with step 1.
4. **Revised Tables 1, 2 and 5** with a before/after comparison against the manuscript (25 July version), and updated per-model JSON records (originals archived in `_archive/pre_A13_measurements/`).

## Pass criterion

For every model: **residual = 0** (no unattributed parameters) and **no parameter counted twice**.

---
## Setup

In [1]:
import os, sys, json, time
from pathlib import Path
import pandas as pd

os.environ.setdefault('HF_HUB_DISABLE_SYMLINKS_WARNING', '1')
os.environ.setdefault('HF_HUB_DISABLE_PROGRESS_BARS', '1')

NOTEBOOK_DIR = Path.cwd()
CODE_ACTIVE = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
sys.path.insert(0, str(CODE_ACTIVE))
MEASUREMENT_DIR = CODE_ACTIVE / 'outputs' / 'measurements'

import transformers, torch
import vlm_audit
from vlm_audit import MODELS, audit_parameter_attribution, trainable_components_for_family
from vlm_audit.checkpoint_headers import read_checkpoint_shapes, split_parameters_and_buffers, attribute_by_prefix
from vlm_audit.logging_utils import sanitize_for_filename

pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)
print('vlm_audit', vlm_audit.__version__, '| transformers', transformers.__version__, '| torch', torch.__version__)
print('Measurement dir:', MEASUREMENT_DIR)

<home>\AppData\Local\Temp\claude\<session>\0edba254-4773-49bc-b009-31568158ee2c\scratchpad\venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


vlm_audit 0.3.0-revision-a13 | transformers 4.57.6 | torch 2.14.1+cpu
Measurement dir: <repository>\code_active\outputs\measurements


### Gated repositories

Two configs sit behind licence gates. They are read from ungated mirrors whose identity with the gated original is verified below, not assumed:

* `meta-llama/Llama-3.2-11B-Vision-Instruct` → `unsloth/Llama-3.2-11B-Vision-Instruct`: the four component counts must equal the counts recorded on the GPU pod from the gated original **to the parameter**.
* `facebook/chameleon-7b` → `FriendliAI/chameleon-7b`: every safetensors shard must have the **same byte size** as the gated original's shard (file sizes are public even for gated repositories).

In [2]:
from huggingface_hub import HfApi
CONFIG_MIRRORS = {
    'meta-llama/Llama-3.2-11B-Vision-Instruct': 'unsloth/Llama-3.2-11B-Vision-Instruct',
    'facebook/chameleon-7b': 'FriendliAI/chameleon-7b',
}

def shard_sizes(repository_id):
    """Return {file_name: size_in_bytes} for the safetensors shards of a repository."""
    information = HfApi().model_info(repository_id, files_metadata=True)
    return {sibling.rfilename: sibling.size for sibling in information.siblings if sibling.rfilename.endswith('.safetensors')}

chameleon_identical = shard_sizes('facebook/chameleon-7b') == shard_sizes('FriendliAI/chameleon-7b')
print('Chameleon mirror shards byte-identical in size to gated original:', chameleon_identical)
assert chameleon_identical

Chameleon mirror shards byte-identical in size to gated original: True


### Published values (manuscript, 25 July version)

Transcribed from Tables 1, 2 and 5 so every change can be shown as before → after.

In [3]:
# name: (total_M, V_M, F_M, L_M, Head_M, V%, F%, L%, V/L, trainable% as printed)
PUBLISHED = {
    'CLIP-ViT-L/14': (427.6, 303.2, 1.4, 123.1, 0.0, 70.9, 0.3, 28.8, 2.46, '100.0'),
    'SigLIP-L/16': (652.5, 316.3, 0.0, 336.2, 0.0, 48.5, 0.0, 51.5, 0.94, '100.0'),
    'SigLIP2-So400m/14': (1136.0, 428.2, 0.0, 707.8, 0.0, 37.7, 0.0, 62.3, 0.61, '100.0'),
    'ViLT-B/32': (117.0, 25.9, 0.0, 85.1, 6.0, 22.1, 0.0, 72.7, 0.31, '100.0'),
    'BLIP-large': (446.3, 303.7, 0.0, 142.6, 0.0, 68.0, 0.0, 32.0, 2.13, '100.0'),
    'BLIP-2-OPT-2.7B': (3744.7, 986.0, 105.1, 2651.7, 2.0, 26.3, 2.8, 70.8, 0.37, '2.8'),
    'InstructBLIP-Vicuna-7B': (7913.7, 986.0, 185.7, 6738.9, 3.1, 12.5, 2.3, 85.2, 0.15, '2.3'),
    'LLaVA-1.5-7B': (7063.4, 303.5, 21.0, 6607.6, 131.3, 4.3, 0.3, 93.5, 0.05, '95.6'),
    'LLaVA-NeXT-7B': (7566.7, 303.5, 21.0, 7110.9, 131.3, 4.0, 0.3, 94.0, 0.04, '95.9'),
    'Molmo-7B-D': (8021.0, 405.0, 0.0, 7616.1, 0.0, 5.0, 0.0, 95.0, 0.05, '95.0'),
    'IDEFICS-9B': (7495.2, 630.8, 126.0, 6607.4, 131.1, 8.4, 1.7, 88.2, 0.10, '~9-19% (base) / 100% (instruct)'),
    'Llama-3.2-11B-Vision-Instruct': (10670.2, 863.6, 31.5, 9249.9, 525.3, 8.1, 0.3, 86.7, 0.09, '~12-20%'),
    'Idefics2-8B': (8402.8, 417.8, 743.2, 7110.7, 131.1, 5.0, 8.8, 84.6, 0.06, '95'),
    'Qwen2-VL-7B-Instruct': (8336.0, 675.8, 44.6, 7070.6, 545.0, 8.1, 0.5, 84.8, 0.10, '100.0'),
    'Qwen2.5-VL-7B-Instruct': (8336.7, 676.6, 44.6, 7070.6, 545.0, 8.1, 0.5, 84.8, 0.10, '100.0'),
    'InternVL3-8B': (8488.0, 304.0, 27.5, 7612.8, 543.6, 3.6, 0.3, 89.7, 0.04, '100.0'),
    'Pixtral-12B': (12682.7, 403.5, 31.5, 11576.7, 671.1, 3.2, 0.2, 91.3, 0.03, '100.0'),
    'DeepSeek-VL2-Small': (16148.3, 428.2, 13.6, 15706.5, 0.0, 2.7, 0.1, 97.3, 0.03, '97.3'),
    'Chameleon-7B': (7311.5, None, None, 6774.7, 536.9, None, None, None, None, '100.0'),
    'Janus-Pro-7B': (7319.6, 388.2, 21.0, 6910.4, 0.0, 5.3, 0.3, 94.4, 0.06, '99.1'),
}
SPEC_BY_NAME = {spec.name: spec for spec in MODELS}
assert set(PUBLISHED) <= set(SPEC_BY_NAME), set(PUBLISHED) - set(SPEC_BY_NAME)

---
## 1. Structure audit — 16 transformers-native models

`audit_parameter_attribution` builds the weightless skeleton, runs the revised extractor, partitions every parameter to one owning component, and applies the documented freezing policy (including the per-model overrides added for IDEFICS-9B and Llama-3.2-Vision; sources in `ModelSpec.training_recipe_note`).

In [4]:
structure_results = {}
for spec in MODELS:
    if spec.name not in PUBLISHED or spec.requires_trust_remote_code:
        continue
    start_time = time.time()
    result = audit_parameter_attribution(spec, config_repository=CONFIG_MIRRORS.get(spec.hf_id))
    structure_results[spec.name] = result
    audit = result['audit']
    print(f"{spec.name:32s} total {audit['model_total']:>15,}  residual {audit['residual']:>6,}  "
          f"overlaps resolved {sum(o['params'] for o in audit['overlaps_resolved']):>12,}  ({time.time() - start_time:.1f}s)")

CLIP-ViT-L/14                    total     427,616,513  residual      0  overlaps resolved            0  (3.1s)


SigLIP-L/16                      total     652,478,466  residual      0  overlaps resolved            0  (0.9s)


BLIP-large                       total     446,290,492  residual      0  overlaps resolved            0  (0.4s)


BLIP-2-OPT-2.7B                  total   3,744,761,856  residual      0  overlaps resolved            0  (0.5s)


InstructBLIP-Vicuna-7B           total   7,913,726,720  residual      0  overlaps resolved            0  (0.7s)


LLaVA-1.5-7B                     total   7,063,427,072  residual      0  overlaps resolved            0  (0.6s)


... (6 outputs left out)

Qwen2.5-VL-7B-Instruct           total   8,292,166,656  residual      0  overlaps resolved   44,574,464  (0.6s)


Pixtral-12B                      total  12,682,739,712  residual      0  overlaps resolved            0  (0.8s)


Llama-3.2-11B-Vision-Instruct    total  10,670,220,835  residual      0  overlaps resolved            0  (0.7s)


Chameleon-7B                     total   7,043,090,432  residual      0  overlaps resolved  268,435,456  (2.7s)


In [5]:
# Llama mirror identity check: component counts must equal the pod measurement of the gated original.
pod_llama = json.loads((CODE_ACTIVE / '_archive' / 'pre_A13_measurements' / 'meta-llama_Llama-3.2-11B-Vision-Instruct.json').read_text())
pod_counts = {name: component['params_total'] for name, component in pod_llama['components'].items()}
mirror_counts = {name: structure_results['Llama-3.2-11B-Vision-Instruct']['component_counts'][name] for name in pod_counts}
print('pod (gated original):', pod_counts)
print('mirror skeleton     :', mirror_counts)
assert pod_counts == mirror_counts, 'Llama mirror differs from the gated original'
print('Llama mirror verified identical to the parameter.')

pod (gated original): {'vision': 863567379, 'head': 525336576, 'language': 9249855504, 'fusion': 31461376}
mirror skeleton     : {'vision': 863567379, 'head': 525336576, 'language': 9249855504, 'fusion': 31461376}
Llama mirror verified identical to the parameter.


---
## 2. Checkpoint audit — 4 remote-code models

Prefix rules mirror the module paths used by `components.py`. The longest matching prefix owns a tensor, so a nested LM head is counted once under `head`. **Molmo is attributed exactly as the published pipeline did** (its `image_projector` connector sits inside `vision_backbone`, so it is counted as vision; that separate misattribution is reviewer comment A8 and is deliberately not changed here).

In [6]:
REMOTE_CODE_PREFIX_RULES = {
    'InternVL3-8B': {'vision_model.': 'vision', 'mlp1.': 'fusion',
                     'language_model.lm_head.': 'head', 'language_model.': 'language'},
    'Molmo-7B-D': {'model.vision_backbone.': 'vision', 'model.transformer.': 'language'},
    'DeepSeek-VL2-Small': {'vision.': 'vision', 'projector.': 'fusion', 'language.': 'language'},
    'Janus-Pro-7B': {'vision_model.': 'vision_und', 'gen_vision_model.': 'vision_gen', 'aligner.': 'fusion',
                     'gen_aligner.': 'fusion_gen', 'gen_embed.': 'fusion_gen', 'gen_head.': 'head_gen',
                     'language_model.': 'language'},
}
LOOSE_TENSOR_RULES = {'image_newline': 'fusion', 'view_seperator': 'fusion'}

checkpoint_shapes = {}
remote_results = {}
for name, prefix_rules in REMOTE_CODE_PREFIX_RULES.items():
    spec = SPEC_BY_NAME[name]
    shapes = read_checkpoint_shapes(spec.hf_id)
    checkpoint_shapes[name] = shapes
    attribution = attribute_by_prefix(shapes, prefix_rules, LOOSE_TENSOR_RULES)
    remote_results[name] = attribution
    residual = attribution['checkpoint_parameter_total'] - attribution['attributed_total']
    print(f"{name:22s} checkpoint {attribution['checkpoint_parameter_total']:>15,}  residual {residual:>6,}  "
          f"components { {k: f'{v/1e6:,.1f}M' for k, v in attribution['component_counts'].items()} }")
    assert residual == 0, attribution['unattributed_by_name']

InternVL3-8B           checkpoint   7,944,373,760  residual      0  components {'language': '7,069.2M', 'vision': '304.0M', 'head': '543.6M', 'fusion': '27.5M'}


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 271afaca-5052-48e4-806f-b512e84cb264)')' thrown while requesting HEAD https://huggingface.co/allenai/Molmo-7B-D-0924/resolve/main/model.safetensors.index.json


Retrying in 1s [Retry 1/5].


Molmo-7B-D             checkpoint   8,021,025,280  residual      0  components {'language': '7,616.1M', 'vision': '405.0M'}


DeepSeek-VL2-Small     checkpoint  16,148,349,504  residual      0  components {'language': '15,706.5M', 'fusion': '13.6M', 'vision': '428.2M'}


Janus-Pro-7B           checkpoint   7,420,434,059  residual      0  components {'vision_und': '316.3M', 'fusion': '21.0M', 'vision_gen': '71.9M', 'fusion_gen': '16.9M', 'head_gen': '83.9M', 'language': '6,910.4M'}


---
## 3. Independent cross-check against the released checkpoints (all 20)

Header totals exclude serialized buffers (`rotary_emb.inv_freq`, `position_ids`). A checkpoint may also store a tied weight twice while the model holds it once; such cases are listed explicitly.

In [7]:
cross_check_rows = []
for name in PUBLISHED:
    spec = SPEC_BY_NAME[name]
    repository_id = CONFIG_MIRRORS.get(spec.hf_id, spec.hf_id)
    shapes = checkpoint_shapes.get(name) or read_checkpoint_shapes(repository_id)
    header_parameters, header_buffers = split_parameters_and_buffers(shapes)
    if name in structure_results:
        audited_total = structure_results[name]['audit']['model_total']
    else:
        audited_total = remote_results[name]['checkpoint_parameter_total']
    cross_check_rows.append({
        'model': name,
        'checkpoint_source': repository_id,
        'published_total_M': PUBLISHED[name][0],
        'audited_total': audited_total,
        'checkpoint_header_total': header_parameters,
        'header_buffers_excluded': header_buffers,
        'header_minus_audited': header_parameters - audited_total,
        'published_minus_audited_M': round(PUBLISHED[name][0] - audited_total / 1e6, 1),
    })
cross_check = pd.DataFrame(cross_check_rows)
cross_check

,model,checkpoint_source,published_total_M,audited_total,checkpoint_header_total,header_buffers_excluded,header_minus_audited,published_minus_audited_M
0,CLIP-ViT-L/14,openai/clip-vit-large-patch14,427.6,427616513,427616513,334,0,-0.0
1,SigLIP-L/16,google/siglip-large-patch16-384,652.5,652478466,652478466,0,0,0.0
2,SigLIP2-So400m/14,google/siglip2-so400m-patch14-384,1136.0,1136008498,1136008498,0,0,-0.0
3,ViLT-B/32,dandelin/vilt-b32-finetuned-vqa,117.0,117588537,117588537,40,0,-0.6
4,BLIP-large,Salesforce/blip-image-captioning-large,446.3,446290492,469732924,512,23442432,0.0
5,BLIP-2-OPT-2.7B,Salesforce/blip2-opt-2.7b,3744.7,3744761856,3744761856,0,0,-0.1
6,InstructBLIP-Vicuna-7B,Salesforce/instructblip-vicuna-7b,7913.7,7913726720,7913726720,0,0,-0.0
7,LLaVA-1.5-7B,llava-hf/llava-1.5-7b-hf,7063.4,7063427072,7063427072,0,0,-0.0
8,LLaVA-NeXT-7B,llava-hf/llava-v1.6-mistral-7b-hf,7566.7,7566747648,7566747648,0,0,-0.0
9,Molmo-7B-D,allenai/Molmo-7B-D-0924,8021.0,8021025280,8021025280,0,0,-0.0


In [8]:
# Any header/audit mismatch must be explained by a tied weight stored twice in the checkpoint.
mismatches = cross_check[cross_check['header_minus_audited'] != 0]
for _, row in mismatches.iterrows():
    shapes = read_checkpoint_shapes(row['checkpoint_source'])
    duplicate_candidates = [(tensor_name, shape) for tensor_name, shape in shapes.items()
                            if int(torch.tensor(shape).prod()) == row['header_minus_audited']]
    print(row['model'], '- extra elements in checkpoint:', f"{row['header_minus_audited']:,}", '-> candidates:', duplicate_candidates)

BLIP-large - extra elements in checkpoint: 23,442,432 -> candidates: [('text_decoder.bert.embeddings.word_embeddings.weight', (30524, 768)), ('text_decoder.cls.predictions.decoder.weight', (30524, 768))]


---
## 4. Revised Tables 1, 2 and 5

Conventions unchanged from the manuscript: V, F, L, Head in millions; percentages over the full checkpoint total; Janus-Pro V aggregates its understanding and generation encoders; Chameleon has no V/L decomposition. `xattn_M` is the inserted cross-attention sub-count **inside** L (new footnote to Table 2).

In [9]:
def collapse_components(name):
    """Return (V, F, L, H, fused, trainable, total, xattn) in raw parameter counts for one model."""
    spec = SPEC_BY_NAME[name]
    if name in structure_results:
        counts = dict(structure_results[name]['component_counts'])
        trainable = dict(structure_results[name]['trainable_counts'])
        audit = structure_results[name]['audit']
        for alias in audit['alias_of']:
            counts.pop(alias, None); trainable.pop(alias, None)
        total = audit['model_total']
        xattn = audit['inserted_cross_attention_params']
    else:
        counts = dict(remote_results[name]['component_counts'])
        policy = dict(trainable_components_for_family(spec.family))
        policy.update(spec.trainable_policy_override or {})
        trainable = {component: (count if policy.get(component, True) is True else 0) for component, count in counts.items()}
        total = remote_results[name]['checkpoint_parameter_total']
        xattn = 0
    vision = counts.get('vision', 0) + counts.get('vision_und', 0) + counts.get('vision_gen', 0)
    fusion = counts.get('fusion', 0) + counts.get('fusion_gen', 0)
    head = counts.get('head', 0) + counts.get('head_gen', 0) + counts.get('embeddings', 0)
    fused = counts.get('fused_transformer', 0)
    language = counts.get('language', 0)
    assert vision + fusion + language + head + fused == total, name
    return vision, fusion, language, head, fused, sum(trainable.values()), total, xattn

revised_rows = []
for name in PUBLISHED:
    vision, fusion, language, head, fused, trainable, total, xattn = collapse_components(name)
    is_native_fused = fused > 0
    revised_rows.append({
        'model': name,
        'total_M': round(total / 1e6, 1),
        'V_M': None if is_native_fused else round(vision / 1e6, 1),
        'F_M': None if is_native_fused else round(fusion / 1e6, 1),
        'L_M': round((fused if is_native_fused else language) / 1e6, 1),
        'Head_M': round(head / 1e6, 1),
        'xattn_M': round(xattn / 1e6, 1) if xattn else 0.0,
        'V_pct': None if is_native_fused else round(100 * vision / total, 1),
        'F_pct': None if is_native_fused else round(100 * fusion / total, 1),
        'L_pct': None if is_native_fused else round(100 * language / total, 1),
        'V_over_L': None if is_native_fused else round(vision / language, 3),
        'trainable_pct_policy': round(100 * trainable / total, 1),
        'total_params': total, 'V_params': vision, 'F_params': fusion, 'L_params': language,
        'Head_params': head, 'fused_params': fused, 'trainable_params': trainable, 'xattn_params': xattn,
    })
revised = pd.DataFrame(revised_rows).set_index('model')
revised[['total_M','V_M','F_M','L_M','Head_M','xattn_M','V_pct','F_pct','L_pct','V_over_L','trainable_pct_policy']]

,total_M,V_M,F_M,L_M,Head_M,xattn_M,V_pct,F_pct,L_pct,V_over_L,trainable_pct_policy
model,,,,,,,,,,,
CLIP-ViT-L/14,427.6,303.2,1.4,123.1,0.0,0.0,70.9,0.3,28.8,2.464,100.0
SigLIP-L/16,652.5,316.3,0.0,336.2,0.0,0.0,48.5,0.0,51.5,0.941,100.0
SigLIP2-So400m/14,1136.0,428.2,0.0,707.8,0.0,0.0,37.7,0.0,62.3,0.605,100.0
ViLT-B/32,117.6,25.9,0.0,85.1,6.6,0.0,22.1,0.0,72.3,0.305,100.0
BLIP-large,446.3,303.7,0.0,142.6,0.0,0.0,68.0,0.0,32.0,2.130,100.0
BLIP-2-OPT-2.7B,3744.8,986.0,105.2,2651.7,2.0,0.0,26.3,2.8,70.8,0.372,2.9
InstructBLIP-Vicuna-7B,7913.7,986.0,185.7,6738.9,3.1,0.0,12.5,2.3,85.2,0.146,2.4
LLaVA-1.5-7B,7063.4,303.5,21.0,6607.6,131.3,0.0,4.3,0.3,93.5,0.046,95.7
LLaVA-NeXT-7B,7566.7,303.5,21.0,7110.9,131.3,0.0,4.0,0.3,94.0,0.043,96.0


In [10]:
# Before -> after, for every printed cell that changes by at least one unit of its printed precision.
PRINTED_FIELDS = [('total_M', 0, 0.1), ('V_M', 1, 0.1), ('F_M', 2, 0.1), ('L_M', 3, 0.1), ('Head_M', 4, 0.1),
                  ('V_pct', 5, 0.1), ('F_pct', 6, 0.1), ('L_pct', 7, 0.1), ('V_over_L', 8, 0.01)]
change_rows = []
for name, published in PUBLISHED.items():
    for field, position, printed_unit in PRINTED_FIELDS:
        before, after = published[position], revised.loc[name, field]
        if before is None or after is None or pd.isna(after):
            continue
        if field == 'V_over_L':  # round the exact ratio once (avoids double rounding of 0.605 -> 0.60)
            rounded_after = round(revised.loc[name, 'V_params'] / revised.loc[name, 'L_params'], 2)
        else:
            rounded_after = round(after, 1)
        if abs(rounded_after - before) >= printed_unit - 1e-9:
            change_rows.append({'model': name, 'field': field, 'published': before, 'revised': rounded_after})
changes = pd.DataFrame(change_rows)
changes

,model,field,published,revised
0,ViLT-B/32,total_M,117.0,117.60
1,ViLT-B/32,Head_M,6.0,6.60
2,ViLT-B/32,L_pct,72.7,72.30
3,BLIP-2-OPT-2.7B,total_M,3744.7,3744.80
4,BLIP-2-OPT-2.7B,F_M,105.1,105.20
5,IDEFICS-9B,total_M,7495.2,8929.70
6,IDEFICS-9B,L_M,6607.4,8041.90
7,IDEFICS-9B,V_pct,8.4,7.10
8,IDEFICS-9B,F_pct,1.7,1.40
9,IDEFICS-9B,L_pct,88.2,90.10


### Trainable share (Table 5) for the two cross-attention models

Computed from the documented recipes (sources quoted in the cell output). Only these two rows are changed by A13; other Table 5 rows are printed for reference, and any difference of 0.1 pp is a rounding/derivation question outside this comment.

In [11]:
for name in ('IDEFICS-9B', 'Llama-3.2-11B-Vision-Instruct'):
    row = revised.loc[name]
    print(f"{name}: trainable {row['trainable_params']:,} of {row['total_params']:,} = {100*row['trainable_params']/row['total_params']:.2f}%")
    print('   recipe:', SPEC_BY_NAME[name].training_recipe_note)
    print(f"   of which inserted cross-attention: {row['xattn_params']:,} ({100*row['xattn_params']/row['L_params']:.1f}% of L)")
pd.DataFrame({'published': {name: values[9] for name, values in PUBLISHED.items()},
              'pipeline_policy': revised['trainable_pct_policy']})

IDEFICS-9B: trainable 1,560,481,552.0 of 8,929,679,632.0 = 17.48%
   recipe: Base checkpoint: 'The pre-trained backbones are frozen while we train the newly initialized parameters' (HuggingFaceM4/idefics-9b model card) -> Perceiver Resampler + 8 gated cross-attention blocks trained; vision encoder, LLaMA-7B and LM head frozen. The -instruct variant unfreezes all parameters (same model card).
   of which inserted cross-attention: 1,434,519,568.0 (17.8% of L)
Llama-3.2-11B-Vision-Instruct: trainable 2,639,926,819.0 of 10,670,220,835.0 = 24.74%
   recipe: Llama 3 paper (arXiv:2407.21783) Sec. 7.4: 'The vision encoder is unfrozen, while the text model weights are kept frozen'; Sec. 7.5 SFT: 'we only update the vision encoder and image adapter weights'. Image adapter = projector + 8 cross-attention layers.
   of which inserted cross-attention: 1,744,898,064.0 (18.9% of L)


,published,pipeline_policy
CLIP-ViT-L/14,100.0,100.0
SigLIP-L/16,100.0,100.0
SigLIP2-So400m/14,100.0,100.0
ViLT-B/32,100.0,100.0
BLIP-large,100.0,100.0
BLIP-2-OPT-2.7B,2.8,2.9
InstructBLIP-Vicuna-7B,2.3,2.4
LLaVA-1.5-7B,95.6,95.7
LLaVA-NeXT-7B,95.9,96.0
Molmo-7B-D,95.0,95.0


---
## 5. Save outputs and update the per-model JSON records

In [12]:
cross_check.to_csv(MEASUREMENT_DIR / 'nb24_parameter_residual_audit.csv', index=False)
revised.reset_index().to_csv(MEASUREMENT_DIR / 'nb24_parameter_components_revised.csv', index=False)
changes.to_csv(MEASUREMENT_DIR / 'nb24_table_changes_vs_manuscript.csv', index=False)
print('Wrote nb24_parameter_residual_audit.csv, nb24_parameter_components_revised.csv, nb24_table_changes_vs_manuscript.csv')

Wrote nb24_parameter_residual_audit.csv, nb24_parameter_components_revised.csv, nb24_table_changes_vs_manuscript.csv


In [13]:
REVISION_NOTE = ('A13 revision: parameters re-attributed with a whole-model residual check '
                 '(every parameter owned by exactly one component); see notebook 24.')

def update_record(name):
    """Write revised parameter counts + audit block into the model's JSON; FLOPs/memory fields are kept."""
    spec = SPEC_BY_NAME[name]
    record_path = MEASUREMENT_DIR / f'{sanitize_for_filename(spec.hf_id)}.json'
    record = json.loads(record_path.read_text(encoding='utf-8'))
    if name in structure_results:
        counts = structure_results[name]['component_counts']
        trainable = structure_results[name]['trainable_counts']
        audit = dict(structure_results[name]['audit'])
    else:
        counts = remote_results[name]['component_counts']
        policy = dict(trainable_components_for_family(spec.family)); policy.update(spec.trainable_policy_override or {})
        trainable = {component: (count if policy.get(component, True) is True else 0) for component, count in counts.items()}
        if name == 'Janus-Pro-7B':
            counts = {**counts, 'vision': counts['vision_und']}; trainable = {**trainable, 'vision': trainable['vision_und']}
        attributed = remote_results[name]['attributed_total']
        audit = {'model_total': remote_results[name]['checkpoint_parameter_total'], 'attributed_total': attributed,
                 'residual': remote_results[name]['checkpoint_parameter_total'] - attributed,
                 'residual_check_passed': remote_results[name]['checkpoint_parameter_total'] == attributed,
                 'unattributed_by_path': remote_results[name]['unattributed_by_name'],
                 'alias_of': {'vision': 'vision_und'} if name == 'Janus-Pro-7B' else {},
                 'inserted_cross_attention_params': 0,
                 'method': 'checkpoint tensor names, longest-prefix attribution (remote-code model)'}
    audit['audited_with'] = f'transformers {transformers.__version__}, meta-device skeleton or checkpoint headers, 2026-10'
    for component_name, count in counts.items():
        component = record['components'].setdefault(component_name, {'name': component_name})
        component['params_total'] = int(count)
        component['params_trainable'] = int(trainable.get(component_name, count))
        if component.get('notes') == 'absent' and count > 0:
            component['notes'] = 'loose top-level parameters only (see parameter_audit)'
    record['parameter_audit'] = audit
    if REVISION_NOTE not in record.get('notes', ''):
        record['notes'] = '\n'.join(part for part in [record.get('notes', ''), REVISION_NOTE] if part)
    record_path.write_text(json.dumps(record, indent=2), encoding='utf-8')
    return record_path.name

for name in PUBLISHED:
    print('updated', update_record(name))

updated openai_clip-vit-large-patch14.json
updated google_siglip-large-patch16-384.json
updated google_siglip2-so400m-patch14-384.json
updated dandelin_vilt-b32-finetuned-vqa.json
updated Salesforce_blip-image-captioning-large.json
updated Salesforce_blip2-opt-2.7b.json
updated Salesforce_instructblip-vicuna-7b.json
updated llava-hf_llava-1.5-7b-hf.json
updated llava-hf_llava-v1.6-mistral-7b-hf.json
updated allenai_Molmo-7B-D-0924.json
updated HuggingFaceM4_idefics-9b.json
updated meta-llama_Llama-3.2-11B-Vision-Instruct.json
updated HuggingFaceM4_idefics2-8b.json
updated Qwen_Qwen2-VL-7B-Instruct.json
updated Qwen_Qwen2.5-VL-7B-Instruct.json
updated OpenGVLab_InternVL3-8B.json
updated mistral-community_pixtral-12b.json
updated deepseek-ai_deepseek-vl2-small.json
updated facebook_chameleon-7b.json
updated deepseek-ai_Janus-Pro-7B.json


### Propagate to the master tables

`master.csv`, `master_wide.csv` and `master_extended.csv` were built by notebook 06 on the GPU pod, which also joins the nb02–nb05 summary CSVs. Re-running notebook 06 would need those (weight-loading) notebooks again, so instead only the **parameter columns** are refreshed from the corrected JSON records, starting from the archived pre-A13 master tables (idempotent). FLOPs, memory and token columns are untouched.

In [14]:
ARCHIVE_DIR = CODE_ACTIVE / '_archive' / 'pre_A13_measurements'
PARAMETER_FIELDS = ('params_total', 'params_trainable')
records_by_id = {}
for spec in MODELS:
    record_path = MEASUREMENT_DIR / f'{sanitize_for_filename(spec.hf_id)}.json'
    if record_path.exists():
        records_by_id[spec.hf_id] = json.loads(record_path.read_text(encoding='utf-8'))

master_long = pd.read_csv(ARCHIVE_DIR / 'master.csv')
new_long_rows = []
for model_id, record in records_by_id.items():
    model_rows = master_long[master_long['model_id'] == model_id]
    if model_rows.empty:
        continue
    for component_name, component in record['components'].items():
        mask = (master_long['model_id'] == model_id) & (master_long['component'] == component_name)
        if mask.any():
            for field in PARAMETER_FIELDS:
                master_long.loc[mask, field] = component.get(field)
        else:
            template = model_rows.iloc[0].to_dict()
            template.update({key: None for key in template if key not in ('model_id','name','family','schema','release_year','architecture_pattern')})
            template.update({'component': component_name, 'params_total': component.get('params_total'),
                             'params_trainable': component.get('params_trainable'), 'notes': component.get('notes', '')})
            new_long_rows.append(template)
master_long = pd.concat([master_long, pd.DataFrame(new_long_rows)], ignore_index=True)
master_long.to_csv(MEASUREMENT_DIR / 'master.csv', index=False)

master_wide = pd.read_csv(ARCHIVE_DIR / 'master_wide.csv')
for model_id, record in records_by_id.items():
    mask = master_wide['model_id'] == model_id
    for component_name, component in record['components'].items():
        for field in PARAMETER_FIELDS:
            column = f'{component_name}__{field}'
            if column not in master_wide.columns:
                master_wide[column] = pd.NA
            master_wide.loc[mask, column] = component.get(field)
master_wide.to_csv(MEASUREMENT_DIR / 'master_wide.csv', index=False)
master_extended = master_wide[master_wide['schema'].isin(['native_fused', 'decoupled_vision'])]
master_extended.to_csv(MEASUREMENT_DIR / 'master_extended.csv', index=False)
print(f'master.csv: {len(master_long)} rows ({len(new_long_rows)} new component rows); '
      f'master_wide.csv: {len(master_wide)} rows, {master_wide.shape[1]} columns; master_extended.csv: {len(master_extended)} rows')

master.csv: 86 rows (2 new component rows); master_wide.csv: 20 rows, 74 columns; master_extended.csv: 2 rows


---
## 6. Summary

In [15]:
passed = (cross_check['header_minus_audited'] == 0).sum()
print(f'Residual check (component sum == whole-model total): passed for all {len(structure_results) + len(remote_results)} models.')
print(f'Audited total == checkpoint header total: {passed}/{len(cross_check)} (mismatches explained in Section 3).')
print()
print('Published totals that differ from the released checkpoint by >= 0.1 M:')
print(cross_check.loc[cross_check['published_minus_audited_M'].abs() >= 0.1,
                      ['model', 'published_total_M', 'audited_total', 'published_minus_audited_M']].to_string(index=False))

Residual check (component sum == whole-model total): passed for all 20 models.
Audited total == checkpoint header total: 19/20 (mismatches explained in Section 3).

Published totals that differ from the released checkpoint by >= 0.1 M:
                 model  published_total_M  audited_total  published_minus_audited_M
             ViLT-B/32              117.0      117588537                       -0.6
       BLIP-2-OPT-2.7B             3744.7     3744761856                       -0.1
            IDEFICS-9B             7495.2     8929679632                    -1434.5
  Qwen2-VL-7B-Instruct             8336.0     8291375616                       44.6
Qwen2.5-VL-7B-Instruct             8336.7     8292166656                       44.5
          InternVL3-8B             8488.0     7944373760                      543.6
          Chameleon-7B             7311.5     7043090432                      268.4
          Janus-Pro-7B             7319.6     7420434059                     -100.8


### Causes of the published discrepancies

| Model | Direction | Cause | Fix |
|---|---|---|---|
| IDEFICS-9B | under-count 1,434.5 M | `model.gated_cross_attn_layers` (8 blocks) not collected by the cross-attention extractor fallback | extractor collects them into L (Sec. 3.2 rule) |
| Janus-Pro-7B | under-count 100.9 M | generation pathway `gen_head`, `gen_aligner`, `gen_embed` never extracted | new `head_gen` / `fusion_gen` slots |
| InternVL3-8B | double-count 543.6 M | `lm_head` counted inside `language_model` and again as head | ownership partition (head owns it) |
| Chameleon-7B | double-count 268.4 M | `embed_tokens` counted inside the fused transformer and again as embeddings | ownership partition |
| Qwen2-VL / Qwen2.5-VL | double-count 44.6 M | `visual.merger` counted inside vision and again as fusion | ownership partition (fusion owns it) |
| ViLT-B/32 | under-count 0.6 M | `vilt.pooler` (feeds only the VQA classifier) and final `vilt.layernorm` not extracted | `UNEXTRACTED_MODULE_COMPONENT` rule: pooler → head, LayerNorm → language |
| BLIP-2-OPT-2.7B | none (rounding) | 3,744,761,856 parameters print as 3,744.8 M, not 3,744.7 M | table regenerated from the record |

BLIP-large's checkpoint file holds 23.4 M more elements than the model because the tied decoder weight (`cls.predictions.decoder.weight` = `word_embeddings.weight`) is serialized twice; the model, and the manuscript, count it once — no change.

Llama-3.2-Vision had **no** missing parameters: its 8 cross-attention layers (1,744.9 M) were already inside L; what changes for Llama is the trainable share (the recipe trains the vision encoder, projector and cross-attention layers). The reviewer's statement that the Perceiver Resampler is absent from Table 2 is not correct: it is the IDEFICS-9B fusion entry (126.0 M).